In [7]:
from utils import *
from utils_nlp import *

from typing import Dict
from collections import defaultdict
import re
import glob
from pathlib import Path


## subject-specific narrative construction [original approach]

In [ ]:
narrative_dir = '../data/narratives'

# snt info
snt_df = pd.read_excel('../data/info/social-navigation-task.xlsx')
snt_df = snt_df[np.isfinite(snt_df['trial_num'])]
snt_df = snt_df.sort_values(by='trial_num')
snt_df = snt_df[snt_df['slide_type'] != 'Game over']
snt_df = snt_df[snt_df['slide_type'] != 'Image']
snt_df.reset_index(drop=True, inplace=True)
snt_df['word_count'] = snt_df['text'].apply(lambda x: len(x.split())) # word count 

# SNT gender versions - right now just choosing one...
gender_versions = [['woman', 'man', 'man', 'man', 'woman', 'woman'], ['man', 'woman', 'woman', 'woman', 'man', 'man']]
genders = gender_versions[0]
character_genders = {'First': genders[0], 'Second': genders[1], 'Assistant': genders[2],
                    'Newcomb':genders[3], 'Hayworth': genders[4], 'Neutral': genders[5]}

#--------------------------------------- helpers for task narrative processing

def replace_pronouns(df: pd.DataFrame, character_genders: Dict[str, str]) -> pd.DataFrame:
    """
    Replace bracketed placeholders in columns ['text','opt1_text','opt2_text']
    using only the provided character_genders mapping.
    """

    required_cols = {'text', 'opt1_text', 'opt2_text'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame missing required columns: {missing}")

    # ---------- helpers ----------
    def forms_for_gender(g: str):
        g = (g or "").strip().lower()
        if g == "woman":
            return dict(
                subj="she", obj="her", poss_det="her", poss_pron="hers", refl="herself",
                is_contr="she's", would_contr="she'd", will_contr="she'll",
                guy_girl="girl", sir_maam="ma'am", man_woman="woman", mr_mrs="Mrs."
            )
        # default to 'man'
        return dict(
            subj="he", obj="him", poss_det="his", poss_pron="his", refl="himself",
            is_contr="he's", would_contr="he'd", will_contr="he'll",
            guy_girl="guy", sir_maam="sir", man_woman="man", mr_mrs="Mr."
        )

    def name_for_role(role: str, gender: str) -> str:
        gender = (gender or "man").lower()
        if role in ("First", "Second"):
            return "Chris" if gender == "man" else "Jessica"
        if role in ("Assistant", "Neutral"):
            return "Anthony" if gender == "man" else "Kayce"
        if role == "Newcomb":
            return "Newcomb"
        if role == "Hayworth":
            return "Hayworth"
        # fallback
        return role

    roles = ["First", "Second", "Assistant", "Newcomb", "Hayworth", "Neutral"]
    role_forms = {r: forms_for_gender(character_genders.get(r, "man")) for r in roles}
    role_names = {r: name_for_role(r, character_genders.get(r, "man")) for r in roles}

    # Newcomb spouse (opposite gender)
    newcomb_gender = (character_genders.get("Newcomb", "man") or "man").lower()
    spouse_gender = "woman" if newcomb_gender == "man" else "man"
    spouse_forms = forms_for_gender(spouse_gender)
    spouse_name = "Mary" if spouse_gender == "woman" else "James"

    # ---------- build replacement rules ----------
    replacements = []

    def pat(token: str) -> re.Pattern:
        return re.compile(re.escape(token))

    for role in roles:
        f = role_forms[role]
        nm = role_names[role]
        mapping = {
            f"[{role} character pronoun]": f["subj"],
            f"[{role} character objective]": f["obj"],
            f"[{role} character reflexive]": f["refl"],
            f"[{role} character possessive pronoun]": f["poss_pron"],
            f"[{role} character possessive determiner]": f["poss_det"],
            f"[{role} character 'is' contraction]": f["is_contr"],
            f"[{role} character 'would' contraction]": f["would_contr"],
            f"[{role} character 'will' contraction]": f["will_contr"],
            f"[Mr./Mrs. {role} character]": f["mr_mrs"],
            f"[{role} character sir/ma'am]": f["sir_maam"],
            f"[{role} character guy/girl]": f["guy_girl"],
            f"[{role} character man/woman]": f["man_woman"],
            f"[{role} character name]": nm,
            f"[{role} character first name]": nm,
        }
        # generic possessive → pronoun possessive determiner, except Newcomb
        if role != "Newcomb":
            mapping[f"[{role} character possessive]"] = f["poss_det"]

        for k, v in mapping.items():
            replacements.append((pat(k), v))

    # Newcomb-specific overrides & spouse
    replacements.append((pat("[Newcomb character possessive]"), "Newcomb's"))
    replacements.append((pat("[Newcomb character spouse's name]"), spouse_name))
    replacements.append((pat("[Newcomb character spouse's pronoun]"), spouse_forms["subj"]))

    # Explicit tokens in your scripts
    replacements.append((pat("[Hayworth character sir/ma'am]"), role_forms["Hayworth"]["sir_maam"]))
    replacements.append((pat("[Second character guy/girl]"), role_forms["Second"]["guy_girl"]))
    replacements.append((pat("[Newcomb character man/woman]"), role_forms["Newcomb"]["man_woman"]))

    # Collapse double spaces after substitution
    space_fix = re.compile(r" {2,}")

    # ---------- build pronoun pattern for capitalization ----------
    pronoun_tokens = set()
    for forms in list(role_forms.values()) + [spouse_forms]:
        for key in ["subj", "obj", "poss_det", "poss_pron", "refl",
                    "is_contr", "would_contr", "will_contr"]:
            pronoun_tokens.add(forms[key])

    # Longer first, escaped, then joined into a single alternation group
    escaped = [re.escape(p) for p in sorted(pronoun_tokens, key=len, reverse=True)]
    pronoun_pattern = r"(?:" + "|".join(escaped) + r")"

    # Start-of-string or start-of-quote pronouns
    sent_start_re = re.compile(
        r"""^([\s"“”']*)(?P<pronoun>""" + pronoun_pattern + r""")\b"""
    )

    # Pronouns after ., !, ? with optional quotes/spaces
    sent_after_punct_re = re.compile(
        r"""([\.!\?]["“”'\s]*)(?P<pronoun>""" + pronoun_pattern + r""")\b"""
    )

    def _capitalize_sentence_initial_pronouns(text: str) -> str:
        # Start of string / quote
        text = sent_start_re.sub(
            lambda m: m.group(1) + m.group("pronoun").capitalize(), text
        )
        # After punctuation
        text = sent_after_punct_re.sub(
            lambda m: m.group(1) + m.group("pronoun").capitalize(), text
        )
        return text

    def _apply(text):
        if not isinstance(text, str):
            return text
        out = text
        # bracket replacements
        for rx, repl in replacements:
            out = rx.sub(repl, out)
        # cleanup spaces
        out = space_fix.sub(" ", out)
        # capitalization at sentence / quote boundaries
        out = _capitalize_sentence_initial_pronouns(out)
        return out

    out = df.copy()
    for col in ["text", "opt1_text", "opt2_text"]:
        out[col] = out[col].map(_apply)

    return out


#--------------------------------------- get the narrative texts for the subjects, based on their decisions

snt_gendered_df = replace_pronouns(snt_df, character_genders) # just choosing an arbitrary version right now

# collect the narratives for each participant, based on their decisions
behav_fnames  = glob.glob(f'{data_dir}/preprocessed/behavior/sub-*.xlsx')
print(f'Found {len(behav_fnames)} behavior files')
for fname in behav_fnames:
    sub_id   = re.search(r"sub-([A-Za-z0-9]+)\.xlsx$", Path(fname).name).group(1)
    behavior = pd.read_excel(fname)

    narrative = []
    for r, snt_slide in snt_gendered_df.iterrows():
        if snt_slide['slide_type'] == 'Decision':
            decision_num = snt_slide['decision_num']
            behav_row    = behavior.iloc[int(decision_num)-1]
            dimension    = behav_row['dimension'] # affil or power
            if dimension == 'neutral': # slightly awkward text
                text = snt_slide['text']
                text = text.replace('1.', '').replace('2.', '').strip()
            else:
                choice_dir    = behav_row[f'{dimension}_decision'] # +/- 1
                dir_to_button = {snt_slide[f'opt1_{dimension}']: 1, snt_slide[f'opt2_{dimension}']: 2}
                text          = snt_slide['opt1_text'] if dir_to_button.get(choice_dir) == 1 else snt_slide['opt2_text']
        else:
            text = snt_slide['text']
        narrative.append(text)

    assert len(narrative) == len(snt_gendered_df)

    narrative_df = pd.DataFrame(narrative, columns=['text'])
    snt_info     = snt_gendered_df[['slide_num', 'slide_type', 'trial_num', 
                                    'narrative_num', 'decision_num', 
                                    'first', 'second', 'assistant', 'powerful', 'boss', 'neutral', 
                                    'character_role_name', 'character_role_num', 'character_decision_num']]
    narrative_df = pd.concat([snt_info.reset_index(drop=True), narrative_df], axis=1)
    narrative_df.to_excel(f'{narrative_dir}/narratives/sub-{sub_id}.xlsx', index=False)  

Found 996 behavior files


## subject-specific narrative construction [trying to standardize wrt gender and names]

In [ ]:
CANONICAL_ROLES  = ["First", "Second", "Assistant", "Powerful", "Boss", "Neutral", "Newcomb", "Hayworth"] # b/c sometimes columns are named differently
ROLE_COL_ALIASES = {
    "first": "First",
    "second": "Second",
    "assistant": "Assistant",
    "powerful": "Powerful",
    "boss": "Boss",
    "neutral": "Neutral",
    "newcomb": "Newcomb",
    "hayworth": "Hayworth",
}

def _find_role_cols(df: pd.DataFrame):
    """Return mapping canonical_role -> existing_column_name (if present)."""
    cols_lower = {c.lower(): c for c in df.columns}
    out = {}
    for k_lower, canonical in ROLE_COL_ALIASES.items():
        if k_lower in cols_lower:
            out[canonical] = cols_lower[k_lower]
    return out

def render_generic_for_target(text: str, target_role: str) -> str:
    """
    Replace bracketed placeholders with:
      - target_role -> "the person" + male pronouns
      - other roles -> "another person" + male pronouns

    Assumes your scripts contain tokens like:
      [First character name], [First character pronoun], ...
    and similarly for other roles.
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""

    # Always-male forms
    male = dict(
        subj="he", obj="him", poss_det="his", poss_pron="his", refl="himself",
        is_contr="he's", would_contr="he'd", will_contr="he'll",
        guy_girl="guy", sir_maam="sir", man_woman="man", mr_mrs="Mr."
    )

    def repl_for(role: str):
        # names: target vs other
        name = "the person" if role == target_role else "another person"

        # for Newcomb/Hayworth spouse tokens, treat spouse as "another person"
        spouse_name = "another person"
        spouse_pron = male["subj"]

        mapping = {
            f"[{role} character pronoun]": male["subj"],
            f"[{role} character objective]": male["obj"],
            f"[{role} character reflexive]": male["refl"],
            f"[{role} character possessive pronoun]": male["poss_pron"],
            f"[{role} character possessive determiner]": male["poss_det"],
            f"[{role} character possessive]": male["poss_det"],

            f"[{role} character 'is' contraction]": male["is_contr"],
            f"[{role} character 'would' contraction]": male["would_contr"],
            f"[{role} character 'will' contraction]": male["will_contr"],

            f"[Mr./Mrs. {role} character]": male["mr_mrs"],
            f"[{role} character sir/ma'am]": male["sir_maam"],
            f"[{role} character guy/girl]": male["guy_girl"],
            f"[{role} character man/woman]": male["man_woman"],

            f"[{role} character name]": name,
            f"[{role} character first name]": name,
        }

        # Newcomb special tokens that appear in your current code
        mapping[f"[Newcomb character spouse's name]"] = spouse_name
        mapping[f"[Newcomb character spouse's pronoun]"] = spouse_pron
        mapping[f"[Newcomb character possessive]"] = "the person's" if role == target_role else "another person's"

        return mapping

    # Build replacements for all roles that might appear as placeholders
    replacements = {}
    for role in ["First", "Second", "Assistant", "Newcomb", "Hayworth", "Neutral", "Powerful", "Boss"]:
        replacements.update(repl_for(role))

    out = text
    for k, v in replacements.items():
        out = out.replace(k, v)

    # cleanup
    out = re.sub(r"\s{2,}", " ", out).strip()
    return out

def resolve_slide_text_for_subject(snt_row: pd.Series, behav_row: pd.Series) -> str:
    """
    Decide which underlying text to use for this subject on this slide.
    This returns the *template text*, still containing bracket placeholders.
    We'll render it per-character later.
    """
    if snt_row["slide_type"] != "Decision":
        return snt_row["text"]

    decision_num = snt_row["decision_num"]
    dimension = behav_row["dimension"]

    if dimension == "neutral":
        # your existing handling
        txt = snt_row["text"]
        return txt.replace("1.", "").replace("2.", "").strip()

    choice_dir = behav_row[f"{dimension}_decision"]  # +/- 1
    dir_to_button = {snt_row[f"opt1_{dimension}"]: 1, snt_row[f"opt2_{dimension}"]: 2}
    chosen = dir_to_button.get(choice_dir)

    if chosen == 1:
        return snt_row["opt1_text"]
    elif chosen == 2:
        return snt_row["opt2_text"]
    else:
        # if something is off, fall back to main decision text
        return snt_row["text"]

def build_subject_character_long_df(
    snt_df: pd.DataFrame,
    behavior: pd.DataFrame,
    *,
    sub_id: str,
) -> pd.DataFrame:
    """
    Returns long df with one row per (slide/trial, involved_character_role),
    including text rendered with target role = 'the person', others = 'another person'.
    """

    role_cols = _find_role_cols(snt_df)  # canonical_role -> column name

    # We will treat these as the roles that define "which character this row belongs to"
    # (exclude Newcomb/Hayworth unless you truly want them as separate relationships)
    relationship_roles = [r for r in ["First", "Second", "Assistant", "Powerful", "Boss", "Neutral"] if r in role_cols]

    rows = []
    for _, snt_row in snt_df.iterrows():
        # Determine the base template text for this subject on this slide
        if snt_row["slide_type"] == "Decision":
            decision_num = int(snt_row["decision_num"])
            behav_row = behavior.iloc[decision_num - 1]
            base_text = resolve_slide_text_for_subject(snt_row, behav_row)
        else:
            base_text = snt_row["text"]

        # Identify which relationship roles are involved in this slide
        involved = []
        for role in relationship_roles:
            col = role_cols[role]
            try:
                if int(snt_row[col]) == 1:
                    involved.append(role)
            except Exception:
                pass

        # If no character is tagged, you can either skip or keep with target_role=None
        if len(involved) == 0:
            continue

        # Create one row per involved character
        for target_role in involved:
            rendered = render_generic_for_target(base_text, target_role)

            rows.append({
                "sub_id": sub_id,
                "target_role": target_role,
                "slide_num": snt_row.get("slide_num", np.nan),
                "trial_num": snt_row.get("trial_num", np.nan),
                "slide_type": snt_row.get("slide_type", ""),
                "narrative_num": snt_row.get("narrative_num", np.nan),
                "decision_num": snt_row.get("decision_num", np.nan),
                "text": rendered,
            })

    return pd.DataFrame(rows)

# # Load and clean snt_df
# snt_df = pd.read_excel("../data/info/social-navigation-task.xlsx")
# snt_df = snt_df[np.isfinite(snt_df["trial_num"])]
# snt_df = snt_df.sort_values(by="trial_num")
# snt_df = snt_df[~snt_df["slide_type"].isin(["Game over", "Image"])]
# snt_df = snt_df.reset_index(drop=True)

# # Subject loop
# narrative_dir = "../data/narratives"
# behav_fnames  = glob.glob(f"{data_dir}/preprocessed/behavior/sub-*.xlsx")
# out_dir       = f"{narrative_dir}/narratives"
# os.makedirs(out_dir, exist_ok=True)
# for fname in behav_fnames:
#     sub_id    = re.search(r"sub-([A-Za-z0-9]+)\.xlsx$", Path(fname).name).group(1)
#     out_fname = f"{out_dir}/sub-{sub_id}_character.xlsx"
#     if os.path.exists(out_fname):
#         continue
#     behavior  = pd.read_excel(fname)
#     long_df   = build_subject_character_long_df(snt_df, behavior, sub_id=sub_id)
#     long_df.to_excel(out_fname, index=False)

## subject-specific embeddings

In [ ]:
#--------------------------------------- embed the full narratives for the subjects, with local and relationship context

from glob import glob
from tqdm import tqdm

def build_decision_contexts(
    narrative_df: pd.DataFrame,
    local_window: int = 1,
    relationship_history="all",  # 0, 1, 2, ... or "all"
):
    """
    For each Decision slide, build:
      - local_context_text: current decision + up to `local_window` preceding non-Decision slides
      - relationship_context_text: local_context_text for this decision + previous decisions with the same character
        controlled by relationship_history:
          "all" -> include all previous decisions for that character
          0     -> include no previous decisions (current only)
          N>0   -> include N previous decisions + current
      - accumulated_text: same as relationship_context_text
    """
    if isinstance(relationship_history, str):
        if relationship_history != "all":
            raise ValueError("relationship_history must be 0, a non-negative int, or 'all'.")
    else:
        relationship_history = int(relationship_history)
        if relationship_history < 0:
            raise ValueError("relationship_history must be 0, a non-negative int, or 'all'.")

    df = narrative_df.reset_index(drop=False).rename(columns={"index": "orig_idx"})
    slide_types = df["slide_type"].tolist()
    decision_idxs = list(df.index[df["slide_type"] == "Decision"])

    local_context_by_idx: dict[int, str] = {}
    local_slides_by_idx: dict[int, list[int]] = {}

    # 1) local context for each decision
    for i in decision_idxs:
        assert slide_types[i] == "Decision", "Index mismatch: expected a Decision slide."

        prev_idxs = []
        k = i - 1
        while k >= 0 and len(prev_idxs) < local_window:
            if slide_types[k] != "Decision":
                prev_idxs.append(k)
            k -= 1

        local_idxs = sorted(prev_idxs) + [i]

        # ---------- ASSERTS on local context ----------
        assert len(local_idxs) >= 1
        assert len(local_idxs) <= local_window + 1
        for j in local_idxs[:-1]:
            assert j < i
            assert slide_types[j] != "Decision"
        assert local_idxs[-1] == i

        local_slides_by_idx[i] = local_idxs
        local_context_by_idx[i] = " ".join(df.loc[local_idxs, "text"])

    # 2) relationship context using histories of local contexts
    char_history: dict[object, list[int]] = defaultdict(list)
    rows = []

    for i in decision_idxs:
        char_id = df.loc[i, "character_role_num"]
        history = char_history[char_id] + [i]

        if relationship_history == "all":
            use_history = history
        else:
            # N previous decisions + current => last (N+1)
            N = relationship_history
            use_history = history[-(N + 1):]

        rel_slide_idxs = []
        for j in use_history:
            rel_slide_idxs.extend(local_slides_by_idx[j])

        # ---------- ASSERTS on relationship context ----------
        assert set(local_slides_by_idx[i]).issubset(set(rel_slide_idxs))
        assert max(rel_slide_idxs) == i
        assert all(j <= i for j in rel_slide_idxs)

        relationship_text = " ".join(local_context_by_idx[j] for j in use_history)
        local_text = local_context_by_idx[i]

        rows.append(
            {
                "local_context_text": local_text,
                "relationship_context_text": relationship_text,
                "accumulated_text": relationship_text,
            }
        )

        char_history[char_id].append(i)

    return pd.DataFrame(rows, index=decision_idxs).sort_index()

# ------------------------ embedding loop 

narrative_dir = "../data/narratives"
llm = "openai"
relationship_history = 0

narr_fnames = glob(f"{narrative_dir}/narratives/sub-*.xlsx")
print(f"Found {len(narr_fnames)} narrative files")

for fname in tqdm(narr_fnames, desc=f"Embedding {llm}"):

    sub_id = re.search(r"sub-([A-Za-z0-9]+)\.xlsx$", Path(fname).name).group(1)
    if relationship_history == "all":
        ctx_tag = "in-context"
    else:
        rh = int(relationship_history)
        if rh == 0:
            ctx_tag = "no-context"
        elif rh > 0:
            ctx_tag = f"{rh:02d}trial-context"
        else:
            raise ValueError("relationship_history must be 0, a non-negative int, or 'all'.")

    out_fname = f"{narrative_dir}/narrative-embeddings/sub-{sub_id}_decisions-{ctx_tag}_{llm}.npz"
    if os.path.exists(out_fname):
        continue

    # build text
    narrative_df = pd.read_excel(fname)
    context_df = build_decision_contexts(
        narrative_df,
        local_window=2,
        relationship_history=relationship_history,
    )
    slides_in_context = context_df["accumulated_text"].to_list()

    # embed text
    semantic_vectors = get_sentence_embeddings(slides_in_context, model=llm, normalize=True)

    # save text and embeddings
    np.savez_compressed(
        out_fname,
        text=np.array(slides_in_context, dtype=object),  # save so can check later
        embedding=semantic_vectors,
    )


Found 112 narrative files


Embedding openai: 100%|██████████| 112/112 [00:54<00:00,  2.05it/s]
